# 📊 Autoportfolio — Notebook Plug-and-Play v1.0

**Sistema Automatizado de Optimización y Re-Optimización de Portafolios**  
Universidad Icesi · CM Project

---

Este notebook es **auto-contenido y ejecutable en Google Colab**.  
Cubre el pipeline completo:

1. 📦 Instalación de dependencias
2. 📥 Descarga y limpieza de datos
3. 🎯 Optimización puntual (MV clásico, robusto, con TC)
4. 🔄 Backtest rolling OOS
5. 📊 Métricas con IC Bootstrap 95%
6. ⚖️ Comparación de estrategias
7. 📈 Dashboard interactivo

> **Reproducibilidad**: `seed=42` usada en todas las celdas relevantes.


## 1. 📦 Instalación de Dependencias

Ejecutar solo si usas **Google Colab** o un entorno limpio.

In [ ]:
# ─── SOLO COLAB / entorno limpio ────────────────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clonar repositorio
    !git clone https://github.com/tu-usuario/autoportfolio.git
    %cd autoportfolio

# Instalar dependencias con versiones fijas
!pip install -q \
    cvxpy==1.8.2 \
    numpy==1.26.4 \
    pandas==2.2.2 \
    scikit-learn==1.5.1 \
    scipy==1.13.1 \
    yfinance==0.2.40 \
    plotly==5.22.0 \
    matplotlib==3.9.0

print('✅ Dependencias instaladas')

In [ ]:
# ─── Imports globales ────────────────────────────────────────────────────────
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Asegura que src/ está en el path
sys.path.insert(0, os.getcwd())

# Semilla global — reproducibilidad (R-07, R-10)
SEED = 42
rng = np.random.default_rng(SEED)

print('✅ Imports OK | seed =', SEED)

## 2. 📥 Descarga y Limpieza de Datos

Configura el **universo de activos** y el **período de análisis** en la celda siguiente.

In [ ]:
# ─── PARÁMETROS — Modificar según necesidad ──────────────────────────────────
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',   # Tech
    'JPM', 'BAC', 'GS',                          # Financiero
    'JNJ', 'PFE',                                # Salud
    'XOM', 'CVX',                                # Energía
    'SPY', 'QQQ', 'TLT',                         # ETFs
]

START_DATE = '2019-01-01'
END_DATE   = '2023-12-31'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
from src.data_client import fetch_prices, fetch_volumes, clean_prices

print(f'Descargando {len(TICKERS)} tickers: {START_DATE} → {END_DATE}...')

prices_raw = fetch_prices(TICKERS, start=START_DATE, end=END_DATE)
volumes_raw = fetch_volumes(TICKERS, start=START_DATE, end=END_DATE)

prices, clean_report = clean_prices(prices_raw)
volumes = volumes_raw.reindex(prices.index).ffill().fillna(1e6)

print(f'✅ Precios: {prices.shape} | Volúmenes: {volumes.shape}')
print(f'   Período: {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'   Tickers disponibles: {list(prices.columns)}')

In [ ]:
# Visualizar precios normalizados
fig, ax = plt.subplots(figsize=(14, 5))
for col in prices.columns:
    norm = prices[col] / prices[col].iloc[0]
    ax.plot(prices.index, norm, label=col, linewidth=0.8, alpha=0.8)
ax.set_title('Precios Normalizados (base = 1.0)')
ax.set_ylabel('Precio normalizado')
ax.legend(ncol=5, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 🎯 Optimización Puntual

Optimización con los últimos 252 días disponibles.

In [ ]:
from src.expected_returns import estimate_expected_returns
from src.covariance_estimators import estimate_covariance
from src.portfolio_optimizer import optimize

# Usar últimos 252 días de datos
window = 252
prices_window = prices.iloc[-window:]

mu    = estimate_expected_returns(prices_window, method='historical')
sigma = estimate_covariance(prices_window, method='ledoit_wolf')

print(f'μ (anualizado): {mu.describe().T}')
print(f'Σ shape: {sigma.shape}')

In [ ]:
# ─── Comparar 3 métodos de optimización ──────────────────────────────────────
# kappa alto (1.0, 3.0) genera diferencias visibles en los pesos
methods = {
    'MV Clásico':      optimize(mu, sigma, method='mv_classic', max_weight=0.30),
    'Robusto κ=1.0':   optimize(mu, sigma, method='robust',     max_weight=0.30, kappa=1.0),
    'Robusto κ=3.0':   optimize(mu, sigma, method='robust',     max_weight=0.30, kappa=3.0),
}

print(f'{"Método":<22} {"Sharpe":>8} {"Max peso":>10} {"N activos>1%":>14} {"Status":>12}')
print('-' * 70)
for name, res in methods.items():
    w = res['weights']
    print(f'{name:<22} {res["expected_sharpe"]:>8.3f} {w.max():>10.1%} {(w > 0.01).sum():>14d} {res["status"]:>12}')

print('\n— Nota: κ alto penaliza activos con μ incierto → portafolio más diversificado (tiende a equal-weight)')


In [ ]:
# Visualizar pesos por método
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, res) in zip(axes, methods.items()):
    w = res['weights'].sort_values(ascending=False)
    w = w[w > 0.001]  # Mostrar solo pesos significativos
    colors = ['steelblue' if v < 0.20 else 'darkorange' for v in w.values]
    ax.barh(w.index, w.values, color=colors)
    ax.set_title(f'{name}\nSharpe={res["expected_sharpe"]:.3f}')
    ax.set_xlabel('Peso')
    ax.axvline(0.30, color='red', linestyle='--', linewidth=0.8, label='Max 30%')
    ax.legend(fontsize=8)
    ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 🔄 Backtest Rolling OOS

Backtest sin look-ahead con ventana deslizante (R-05).

In [ ]:
from src.backtest_engine import BacktestConfig, run_backtest, run_comparison

# ─── PARÁMETROS DEL BACKTEST — Modificar según necesidad ─────────────────────
config = BacktestConfig(
    opt_method      = 'mv_classic',   # Método de optimización
    rebalance_freq  = 'Q',            # 'M' mensual | 'Q' trimestral
    window          = 252,            # Días de entrenamiento
    warmup          = 252,            # Días de warmup inicial
    mu_method       = 'historical',   # 'historical' | 'capm_shrunk'
    cov_method      = 'ledoit_wolf',  # 'sample' | 'ledoit_wolf' | 'pca'
    max_weight      = 0.30,           # Peso máximo por activo (R-04)
    turnover_limit  = 0.20,           # Límite turnover por rebalanceo (R-02)
    commission_bps  = 5.0,            # Comisión en bps (R-01)
    spread_bps      = 2.0,            # Bid-ask spread en bps (R-01)
    impact_coef     = 0.10,           # Market impact TWAP (R-01)
    seed            = SEED,           # Reproducibilidad (R-07)
)
# ─────────────────────────────────────────────────────────────────────────────

print('Ejecutando backtest...')
result = run_backtest(prices, volumes, config)

print(f'✅ Backtest completo')
print(f'   Período: {result.nav.index[0].date()} → {result.nav.index[-1].date()}')
print(f'   Días:    {len(result.nav)}')
print(f'   NAV inicial: ${result.nav.iloc[0]:,.0f}')
print(f'   NAV final:   ${result.nav.iloc[-1]:,.0f}')
print(f'   Rebalanceos: {len(result.rebalance_log)}')

In [ ]:
# Visualizar NAV y drawdown
nav_norm = result.nav / result.nav.iloc[0]
peak = nav_norm.cummax()
drawdown = (nav_norm - peak) / peak

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(nav_norm.index, nav_norm.values, color='steelblue', linewidth=1.5, label='Portafolio')
ax1.set_title('NAV Normalizado')
ax1.set_ylabel('NAV (base=1)')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.fill_between(drawdown.index, drawdown.values, 0, color='crimson', alpha=0.5)
ax2.set_title('Drawdown')
ax2.set_ylabel('Drawdown')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 📊 Métricas con IC Bootstrap 95%

Calcula todas las métricas requeridas (R-08) con intervalo de confianza bootstrap (R-06).

In [ ]:
from src.metrics_calculator import compute_metrics, metrics_to_dict

# Construir serie de turnover
turnover_s = pd.Series({r.date: r.turnover for r in result.daily_records})

# Calcular métricas — seed=42 para bootstrap reproducible (R-06, R-07)
metrics = compute_metrics(
    returns         = result.returns,
    nav             = result.nav,
    turnover        = turnover_s,
    risk_free_rate  = 0.0,
    n_bootstrap     = 1000,   # IC 95% con 1000 iteraciones (R-06)
    seed            = SEED,   # Reproducibilidad (R-07)
)

print('=' * 50)
print('  MÉTRICAS DE RENDIMIENTO (R-08)')
print('=' * 50)
print(f'  Retorno total:        {metrics.total_return:>10.2%}')
print(f'  Retorno anualizado:   {metrics.annualized_return:>10.2%}')
print(f'  Volatilidad anual:    {metrics.annualized_volatility:>10.2%}')
print(f'  Sharpe ratio:         {metrics.sharpe_ratio:>10.3f}')
print(f'  Sharpe IC 95%:        [{metrics.sharpe_ci_low:.3f}, {metrics.sharpe_ci_high:.3f}]')
print(f'  Sortino ratio:        {metrics.sortino_ratio:>10.3f}')
print(f'  Calmar ratio:         {metrics.calmar_ratio:>10.3f}')
print(f'  Maximum Drawdown:     {metrics.max_drawdown:>10.2%}')
print(f'  VaR 95% histórico:    {metrics.var_95_historical:>10.4f}')
print(f'  VaR 95% paramétrico:  {metrics.var_95_parametric:>10.4f}')
print(f'  CVaR 95%:             {metrics.cvar_95:>10.4f}')
print(f'  Beta vs benchmark:    {metrics.beta:>10.3f}')
print(f'  Turnover anual prom:  {metrics.avg_annual_turnover:>10.2%}')
print(f'  N rebalanceos:        {metrics.n_rebalances:>10d}')
print('=' * 50)

## 6. ⚖️ Comparación de Estrategias

Tres configuraciones **estructuralmente distintas** para ver diferencias reales:

| Config | Qué hace diferente |
|--------|-------------------|
| **MV Clásico** | Maximiza Sharpe puro — concentrado en los mejores activos |
| **Robusto κ=2.0** | Penaliza fuerte la incertidumbre en μ → más diversificado, menor vol |
| **Con TC λ=0.01** | Penaliza el turnover en el objetivo → portafolio más estable entre rebalanceos |


In [ ]:
from src.metrics_calculator import compare_metrics

# ─── Configuraciones estructuralmente distintas ───────────────────────────────
configs = {
    # Baseline: maximiza Sharpe puro, sin penalizaciones
    'MV Clásico': BacktestConfig(
        opt_method='mv_classic', window=252, warmup=252,
        rebalance_freq='QE', max_weight=0.30, seed=SEED,
    ),
    # Robusto con kappa alto: penaliza fuerte la incertidumbre en μ
    # → reduce concentración, baja volatilidad, menor Sharpe esperado
    'Robusto κ=2.0': BacktestConfig(
        opt_method='robust', window=252, warmup=252,
        rebalance_freq='QE', max_weight=0.30, kappa=2.0, seed=SEED,
    ),
    # Con penalización de TC: el optimizer ya descuenta los costos de rebalanceo
    # → portafolio más estable (menor turnover), menor drag por costos
    'MV + TC λ=0.01': BacktestConfig(
        opt_method='mv_tc', window=252, warmup=252,
        rebalance_freq='QE', max_weight=0.30, tc_lambda=0.01,
        commission_bps=5.0, spread_bps=2.0, seed=SEED,
    ),
}
# ─────────────────────────────────────────────────────────────────────────────

print('Ejecutando backtests...')
results = run_comparison(prices, volumes, configs)
print(f'✅ {len(results)} backtests completados')


In [ ]:
# Calcular métricas para cada estrategia
all_metrics = {}
for label, res in results.items():
    to_s = pd.Series({r.date: r.turnover for r in res.daily_records})
    all_metrics[label] = compute_metrics(
        res.returns, res.nav, turnover=to_s,
        n_bootstrap=1000, seed=SEED  # R-06, R-07
    )

# Tabla comparativa
table = compare_metrics(all_metrics)
display_rows = [
    'annualized_return', 'annualized_volatility', 'sharpe_ratio',
    'sharpe_ci_low', 'sharpe_ci_high', 'sortino_ratio',
    'max_drawdown', 'calmar_ratio', 'var_95_historical',
    'cvar_95', 'avg_annual_turnover'
]
print(table.loc[display_rows].round(4).T.to_string())

In [ ]:
# Comparación de NAV normalizado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NAV curves
ax = axes[0]
for label, res in results.items():
    nav_n = res.nav / res.nav.iloc[0]
    ax.plot(nav_n.index, nav_n.values, label=label, linewidth=1.5)
ax.set_title('NAV Comparado (Normalizado)')
ax.set_ylabel('NAV (base=1)')
ax.legend()
ax.grid(alpha=0.3)

# Sharpe con IC 95% (R-06)
ax2 = axes[1]
labels = list(all_metrics.keys())
sharpes = [all_metrics[l].sharpe_ratio for l in labels]
ci_lows = [all_metrics[l].sharpe_ci_low for l in labels]
ci_highs = [all_metrics[l].sharpe_ci_high for l in labels]
yerr = [[s - lo for s, lo in zip(sharpes, ci_lows)],
        [hi - s for s, hi in zip(sharpes, ci_highs)]]
ax2.bar(labels, sharpes, yerr=yerr, capsize=8, alpha=0.75,
        color=['steelblue', 'darkorange', 'green'])
ax2.set_title('Sharpe con IC 95% Bootstrap (R-06)')
ax2.set_ylabel('Sharpe Ratio')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 📈 Dashboard Interactivo

Genera un HTML con NAV curves, frontera eficiente, heatmap de pesos y scatter turnover vs Sharpe.

In [ ]:
from src.dashboard_results import build_dashboard

output_path = 'results/dashboard_notebook.html'
os.makedirs('results', exist_ok=True)

build_dashboard(
    results,
    output_path=output_path,
    title='Autoportfolio — Comparación de Estrategias',
)

print(f'✅ Dashboard generado: {output_path}')

# En Colab: descargar automáticamente
if IN_COLAB:
    from google.colab import files
    files.download(output_path)
else:
    print('   Abre el archivo HTML en tu navegador para ver el dashboard interactivo.')

## 8. 🔧 Sección de Personalización

Modifica los parámetros clave y re-ejecuta todo el notebook.

In [ ]:
# ─── PARÁMETROS CONFIGURABLES — Modificar a gusto ────────────────────────────
CUSTOM_CONFIG = BacktestConfig(
    opt_method      = 'robust',        # 'mv_classic' | 'robust' | 'mv_tc' | 'mv_turnover'
    rebalance_freq  = 'Q',             # 'M' | 'Q'
    window          = 252,             # Ventana de entrenamiento en días
    warmup          = 252,             # Warmup en días
    mu_method       = 'capm_shrunk',   # 'historical' | 'capm_shrunk' | 'black_litterman'
    cov_method      = 'ledoit_wolf',   # 'sample' | 'ledoit_wolf' | 'pca'
    replacement_rule= 'momentum',      # 'none' | 'momentum' | 'random' | 'min_variance'
    max_weight      = 0.25,            # Peso máximo por activo
    turnover_limit  = 0.15,            # Límite de turnover
    kappa           = 0.10,            # Nivel de robustez
    commission_bps  = 5.0,
    spread_bps      = 2.0,
    impact_coef     = 0.10,
    seed            = SEED,            # R-07: Reproducibilidad
)
# ─────────────────────────────────────────────────────────────────────────────

custom_result = run_backtest(prices, volumes, CUSTOM_CONFIG)

to_s = pd.Series({r.date: r.turnover for r in custom_result.daily_records})
custom_metrics = compute_metrics(
    custom_result.returns, custom_result.nav,
    turnover=to_s, n_bootstrap=1000, seed=SEED
)

print(f'Sharpe:  {custom_metrics.sharpe_ratio:.3f}  IC=[{custom_metrics.sharpe_ci_low:.3f}, {custom_metrics.sharpe_ci_high:.3f}]')
print(f'MaxDD:   {custom_metrics.max_drawdown:.2%}')
print(f'Ann.Ret: {custom_metrics.annualized_return:.2%}')
print(f'Turnover anual: {custom_metrics.avg_annual_turnover:.2%}')

---

## ✅ Restricciones Verificadas

| ID | Restricción | Módulo |
|----|-------------|--------|
| R-01 | Costos de TC explícitos (comisión + spread + TWAP impact) | `optimizer_with_tc`, `order_execution` |
| R-02 | Turnover limitado (hard cap + blending) | `turnover_constraint`, `rebalancing_engine` |
| R-03 | Liquidez Amihud + bounds por ADV | `liquidity_constraint`, `universe_filter` |
| R-04 | Pesos ≥0, suma=1, máx 30% | Todos los optimizadores |
| R-05 | Sin look-ahead — rolling OOS estricto | `rolling_engine`, `backtest_engine` |
| R-06 | Bootstrap IC 95% Sharpe (1000 iter.) | `metrics_calculator.bootstrap_sharpe_ci` |
| R-07 | `seed=42` global en todo el pipeline | `BacktestConfig.seed`, `rng` |
| R-08 | Métricas completas (Sharpe, Sortino, MaxDD, Calmar, VaR, CVaR, Beta, IR) | `metrics_calculator` |
| R-09 | Sin overfitting (Ledoit-Wolf, OOS, bootstrap) | `covariance_estimators`, `backtest_engine` |
| R-10 | Pipeline modular, auto-contenido, reproducible | Toda la arquitectura |

---
*Autoportfolio v1.0 · Universidad Icesi · seed=42*